# Multimodal Visual RAG: Colab GPU smoke test
Only Qwen3-VL-2B, float16, two synthetic images. No benchmark download or full suite.
Select **Runtime → Change runtime type → T4 GPU** first. Run cells in order.
The model download is approximately 4.3 GB; running this notebook authorizes that remote download.
Build the source bundle locally with `python3 scripts/build_colab_bundle.py` after committing.
Upload its ZIP here and copy the commit/checksum from `colab_outputs/source/manifest.json`.
Do not upload credentials. The source checksum must be copied independently, not trusted from an uploaded ZIP.


In [ ]:
import platform, subprocess
print(platform.python_version())
subprocess.run(["nvidia-smi"], check=True)
# Stop here if no GPU is allocated. No CPU or MPS experiment is supported.


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
from pathlib import Path
OUTPUT = Path('/content/drive/MyDrive/mmrag/colab_outputs/smoke_test')
OUTPUT.mkdir(parents=True, exist_ok=True)
probe = OUTPUT / 'storage_probe.txt'
probe.write_text('persistent storage read-back')
assert probe.read_text() == 'persistent storage read-back'
# Leave Drive mounted to retain reports even if subsequent cells fail.


In [ ]:
SOURCE_COMMIT = ""  # paste full 40-character SHA from the local manifest
EXPECTED_SHA256 = ""  # paste checksum from the local manifest
assert len(SOURCE_COMMIT) == 40 and len(EXPECTED_SHA256) == 64
uploaded = files.upload()  # choose ONLY mmrag-<commit>.zip
assert len(uploaded) == 1
import hashlib, io, zipfile
payload = next(iter(uploaded.values()))
assert hashlib.sha256(payload).hexdigest() == EXPECTED_SHA256, "Source checksum mismatch"
SOURCE = Path('/content') / ('mmrag-' + SOURCE_COMMIT)
SOURCE.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for item in archive.infolist():
        target = (SOURCE / item.filename).resolve()
        assert target.is_relative_to(SOURCE.resolve()), "Unsafe archive path"
        assert (item.external_attr >> 16) & 0o170000 != 0o120000, "Symlinks prohibited"
    archive.extractall(SOURCE)
assert (SOURCE / 'uv.lock').exists()
import json
(OUTPUT / 'source_provenance.json').write_text(json.dumps({
    'source_commit': SOURCE_COMMIT, 'bundle_sha256': EXPECTED_SHA256}, indent=2))


In [ ]:
import sys, shutil, subprocess
# uv.lock pins project dependencies; an isolated Python 3.12 environment leaves
# Colab's preinstalled packages intact. CUDA compatibility is tested, not assumed.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.11.16'], check=True)
UV = shutil.which('uv')
subprocess.run([UV, 'sync', '--frozen', '--python', '3.12', '--extra', 'models'],
               cwd=SOURCE, check=True)
PYTHON = str(SOURCE / '.venv/bin/python')
subprocess.run([PYTHON, '-c',
    "import torch, transformers; print(torch.__version__, transformers.__version__, "
    "torch.version.cuda, torch.cuda.is_available()); assert torch.cuda.is_available()"], check=True)


In [ ]:
# Runs only the production VLM backend, never the Kaggle runner.
command = [PYTHON, '-m', 'mmrag.colab_smoke_test', '--source-commit', SOURCE_COMMIT,
           '--output', str(OUTPUT), '--runtime-type', 'Google Colab GPU']
with (OUTPUT / 'console.log').open('a') as log:
    process = subprocess.Popen(command, cwd=SOURCE, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end=''); log.write(line); log.flush()
    returncode = process.wait()
print('exit status:', returncode)
reports = sorted(OUTPUT.glob('*/report.json'), key=lambda p: p.stat().st_mtime)
assert reports, 'No report produced; inspect console.log and installation cell'
report_path = reports[-1]
report = json.loads(report_path.read_text())
expected = report_path.with_suffix('.sha256').read_text().split()[0]
assert hashlib.sha256(report_path.read_bytes()).hexdigest() == expected
for artifact in report['outputs']:
    assert hashlib.sha256(Path(artifact['path']).read_bytes()).hexdigest() == artifact['sha256']
print(json.dumps(report, indent=2))
assert returncode == 0 and report['status'] == 'passed', 'Smoke test failed; preserve report'


## Return the evidence
Download the latest run directory (report.json, report.sha256, example JSONs) plus source_provenance.json and console.log from Drive. A pass tests execution only, not answer accuracy or full-suite feasibility. Do not start 4B/8B or the full runner from this notebook.